# Domain-wall bipartite mutual-information sweep

This Colab/A100 notebook measures the trajectory-resolved mutual information $I_{a,b}=S_a+S_b-S_{a\cup b}$ between two opposite full-$x$ strips of width $w=N_y/4$. It averages the nonlinear observable over strip translations and then over 100 independent trajectories for $N_y=20,24,28$, 21 values of $\alpha_1$, and hard/soft wall constructions. It reuses verified five-sample v1 results and computes only missing trajectories in 50/25-sample GPU batches. Covariance matrices are materialized only for the endpoint observer and are never saved.

## 1. Runtime setup

Select an **A100 GPU** runtime before continuing. Google Drive is mounted once; the executable bundle and transient work are copied to local `/content`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Complete campaign configuration

These are the locked v2 production values. `LEGACY_OUTPUT_ROOT` is read-only and points to the original five-sample run. `REPORT_ONLY=True` prints the verified v2 and reusable-v1 inventory without running GPU work. Set `MAX_NEW_TASKS` to a positive integer to limit one Colab session.

In [ ]:
from pathlib import Path

BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs/02_domain_wall_bipartite_mutual_information')
LEGACY_OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/domain_wall_bmi_nx20_ny20-28_alpha21_desc_c2ny_s100_v1')
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs/domain_wall_bmi_nx20_ny20-28_alpha21_desc_c2ny_s100_v2_batched_50-25-25')
LOCAL_BUNDLE_DIR = Path('/content/02_domain_wall_bipartite_mutual_information')
SCRATCH_ROOT = Path('/content/domain_wall_bmi_scratch')
REPORT_ONLY = False
MAX_NEW_TASKS = None

CONFIG = {
    'sampling_revision': 'domain_wall_bmi_nx20_ny20-28_alpha21_desc_c2ny_s100_v2_batched_50-25-25',
    'root_seed': 2026090301,
    'Nx': 20,
    'Ny_values': [20, 24, 28],
    'width_rule': 'Ny//4',
    'alpha_1_values': [
        3.0, 2.75, 2.5, 2.3, 2.2, 2.15, 2.1, 2.075, 2.05, 2.025,
        2.0, 1.975, 1.95, 1.925, 1.9, 1.85, 1.8, 1.7, 1.5, 1.25, 1.0,
    ],
    'alpha_2': 30.0,
    'wall_constructions': ['hard', 'soft'],
    'nshell': 1,
    'samples_per_case': 100,
    'batch_size_by_Ny': {'20': 50, '24': 25, '28': 25},
    'cycles_rule': '2*Ny',
    'device': 'cuda:0',
    'dtype': 'complex128',
    'protocol': {
        'DW': True,
        'filling_frac': 0.5,
        'trial_orbitals': 'X',
        'init_mode': 'default',
        'sequence': 'raster_y',
        'perfect_correction': True,
        'postselect': False,
        'postselect_probability': 0.0,
        'meas_slab_only': True,
        'n_a': 0.5,
        'triv_region_local_mode': False,
    },
}

## 3. Stage locally and run or resume

The runner verifies every v2 result/completion pair before skipping it and imports only checksum-valid v1 pairs. The outer progress bar reports completed, skipped, imported, pending, and failed macro-tasks; the canonical engine reports cycle progress and the observer reports endpoint translation progress. An interruption reruns only the active macro-task.

In [ ]:
import codecs
import json
import os
import shutil
import subprocess
import sys

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(f'Missing Drive bundle: {BUNDLE_DRIVE_DIR}')
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
runner_path = LOCAL_BUNDLE_DIR / 'run_campaign.py'
runner_source = runner_path.read_text(encoding='utf-8')
expected_revision = CONFIG['sampling_revision']
if '--legacy-output-root' not in runner_source or expected_revision not in runner_source:
    raise RuntimeError(
        'The Drive bundle is stale: this v2 notebook was paired with an older '
        'run_campaign.py. Re-upload the complete local bundle directory—not only '
        'the notebook—then restart this cell. Staged runner: ' + str(runner_path)
    )
config_path = Path('/content/domain_wall_bmi_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')

command = [
    sys.executable, '-u', str(runner_path),
    '--config', str(config_path),
    '--output-root', str(OUTPUT_ROOT),
    '--legacy-output-root', str(LEGACY_OUTPUT_ROOT),
    '--scratch-root', str(SCRATCH_ROOT),
]
if REPORT_ONLY:
    command.append('--report-only')
if MAX_NEW_TASKS is not None:
    command.extend(['--max-new-tasks', str(int(MAX_NEW_TASKS))])
print('[launch]', ' '.join(command), flush=True)
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=0,
)
if process.stdout is None:
    raise RuntimeError('Runner output pipe was not created')
decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
while True:
    raw = os.read(process.stdout.fileno(), 4096)
    if not raw:
        break
    sys.stdout.write(decoder.decode(raw))
    sys.stdout.flush()
tail = decoder.decode(b'', final=True)
if tail:
    sys.stdout.write(tail)
    sys.stdout.flush()
returncode = process.wait()
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)
print('[notebook] runner exited successfully', flush=True)

## 4. Final figure and completion status

The figure exists only after all 420 v2 macro-tasks and the consolidated products pass checksum verification. A partial session prints the missing-figure status without treating the campaign as complete.

In [ ]:
import json
from IPython.display import Image, display

aggregate_root = OUTPUT_ROOT / 'aggregate'
completion_path = aggregate_root / 'aggregate.complete.json'
figure_path = aggregate_root / 'domain_wall_bipartite_mutual_information.png'
if completion_path.is_file() and figure_path.is_file():
    completion = json.loads(completion_path.read_text(encoding='utf-8'))
    print(json.dumps(completion, indent=2, sort_keys=True))
    display(Image(filename=str(figure_path)))
else:
    print('[notebook] aggregate figure is pending; rerun to continue the campaign')

## 5. Release the runtime

Run this final cell after the runner exits successfully, including intentionally bounded partial sessions.

In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')